https://adk.dev/agents/workflow-agents/


Además de la típica arquitectura ReAct, Google ADK nos ofrece tres tipos de arquitecturas multiagente adicionales

# Supervisor Agents

Un agente supervisor es aquel agente ReAct que en lugar de tener funciones como herramientas, tiene otros agentes como herramientas

In [ ]:
from google.adk.agents import LlmAgent

agente_1 = LlmAgent(...)
agente_2 = LlmAgent(...)
...
agente_n = LlmAgent(...)

agente_supervisor = LlmAgent(
    # Parámetros habituales...
    sub_agents = [agente_1, agente_2,..., agente_n]
)

Otra forma es pasar los sub_agents como tools

In [ ]:
# Conceptual Setup: Agent as a Tool
from google.adk.agents import LlmAgent, BaseAgent
from google.adk.events import Event
from google.adk.tools import agent_tool
from pydantic import BaseModel
from google.genai import types


# Define a target agent (could be LlmAgent or custom BaseAgent)
class ImageGeneratorAgent(BaseAgent): # Example custom agent
    name: str = "ImageGen"
    description: str = "Generates an image based on a prompt."
    # ... internal logic ...
    async def _run_async_impl(self, ctx): # Simplified run logic
        prompt = ctx.session.state.get("image_prompt", "default prompt")
        # ... generate image bytes ...
        image_bytes = b"..."
        yield Event(author=self.name, content=types.Content(parts=[types.Part.from_bytes(image_bytes, "image/png")]))


image_agent = ImageGeneratorAgent()
image_tool = agent_tool.AgentTool(agent=image_agent) # Wrap the agent


# Parent agent uses the AgentTool
artist_agent = LlmAgent(
    name="Artist",
    model="gemini-flash-latest",
    instruction="Create a prompt and use the ImageGen tool to generate the image.",
    tools=[image_tool] # Include the AgentTool
)
# Artist LLM generates a prompt, then calls:
# FunctionCall(name='ImageGen', args={'image_prompt': 'a cat wearing a hat'})
# Framework calls image_tool.run_async(...), which runs ImageGeneratorAgent.
# The resulting image Part is returned to the Artist agent as the tool result.

# Sequential Agents

Como su nombre indica, los agentes se ejecutan de manera secuencial

In [ ]:
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.llm_agent import LlmAgent

agente_1 = LlmAgent(...)
agente_2 = LlmAgent(...)
...
agente_n = LlmAgent(...)

agente_sequencial = SequentialAgent(
    # Parámetros habituales...
    sub_agents = [agente_1, agente_2,..., agente_n]
)

#  Loop Agents

Ejecuta sus sub agentes repetidamente hasta que una condición se cumpla

In [ ]:
from google.adk.agents.loop_agent import LoopAgent
from google.adk.agents.llm_agent import LlmAgent

agente_1 = LlmAgent(...)
agente_2 = LlmAgent(...)
...
agente_n = LlmAgent(...)

agente_loop = LoopAgent(
    # Parámetros habituales...
    sub_agents = [agente_1, agente_2,..., agente_n],
    max_iterations = 5
)

# Parallel Agents

Ejecuta sus sub agentes de manera paralela

In [ ]:
from google.adk.agents.parallel_agent import ParallelAgent

from google.adk.agents.loop_agent import LoopAgent
from google.adk.agents.llm_agent import LlmAgent

agente_1 = LlmAgent(...)
agente_2 = LlmAgent(...)
...
agente_n = LlmAgent(...)

agente_loop = ParallelAgent(
    # Parámetros habituales...
    sub_agents = [agente_1, agente_2,..., agente_n]
)

# Importancia del State

Los distintos subagentes se comunican entre sí o con un agente superior via el estado